In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
# Create SparkSession
spark =  SparkSession.builder \
                    .master("spark://spark-master:7077") \
                    .appName("example") \
                    .config("spark.executor.memory", "2g") \
                    .getOrCreate()
# spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 10:32:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext

In [4]:
data = [1, 2, 3, 4, 5]
distData = sc.parallelize(data,10)

# distData.cache()

print(distData)
distData.collect()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299


[1, 2, 3, 4, 5]

## Transformations on RDDs

### Transformations (Lazy Operations)
Transformations create a new RDD from an existing one and are **evaluated lazily** (execution happens only when an action is called).
#### Basic Transformations
- `map` - `filter` - `flatMap` - `mapPartitions`
#### Set Operations
- `union` - `intersection` - `distinct`
#### Key-Value Transformations
- `groupByKey` - `reduceByKey` - `aggregateByKey`
#### RDD Combination Operations
- `join`- `cogroup`- `cartesian`
#### Sampling Operations
- `sample`- `takeSample`
---
### Actions (Eager Operations)
Actions trigger execution and return results to the driver or write output to storage
#### Data Retrieval Actions
- `collect`- `count`- `first`- `take`
#### Aggregation Actions
- `reduce`- `fold`- `aggregate`
#### Output & Side-Effect Actions
- `foreach`- `foreachPartition`
#### Save Operations
- `saveAsTextFile`- `saveAsSequenceFile`
#### Key-Value Actions
- `countByKey`- `lookup`

#### Narrow Transforamtions

In [5]:
#Using map() transformation we take in any function, and that function is applied to every element of RDD.
#👉 In PySpark, map() is internally implemented using mapPartitions()
#👉 Because execution involves Python code, Spark wraps it as PythonRDD.
squared = distData.map(lambda x: x *2)  # [1, 4, 9, 16, 25]
squared

PythonRDD[1] at RDD at PythonRDD.scala:58

In [8]:
#The flatMap transformation applies a function to each element of an RDD and returns a new RDD where each input element can be mapped to zero or more output elements such as tokenizing text or exploding arrays. 
lines = sc.parallelize(["hello worl", "hi"])
words = lines.flatMap(lambda line: line.split(" "))
words

PythonRDD[3] at RDD at PythonRDD.scala:58

#### Wide Transformations

In [4]:
#reduceByKey() is an efficient transformation for combining values with the same key using an associative and commutative reduce function. It operates in two stages:
#Example 1
data = [('A', 1), ('B', 2), ('A', 3), ('B', 4), ('A', 5)]
rdd = sc.parallelize(data)

# Apply reduceByKey
result = rdd.reduceByKey(lambda a, b: a + b)
print(result)
result.collect()

# #Example 2
# pairs = sc.parallelize([("apple", 1), ("banana", 1), ("apple", 1)])
# word_counts = pairs.reduceByKey(lambda a, b: a + b) 

PythonRDD[5] at RDD at PythonRDD.scala:58


[('B', 6), ('A', 9)]

In [5]:
result.glom().collect()

[[], [('B', 6)], [], [], [], [], [('A', 9)], []]

In [28]:
#groupByKey() collects all values associated with each key into a single iterable. Unlike, it doesn't perform any aggregation.
data = [('A', 1), ('B', 2), ('A', 3), ('B', 4), ('A', 5)]
rdd = sc.parallelize(data)

grouped = rdd.groupByKey()

print(grouped)
print(grouped.collect())

result = grouped.mapValues(list)
result.collect()

PythonRDD[65] at RDD at PythonRDD.scala:58
[('B', <pyspark.resultiterable.ResultIterable object at 0x7efec4d35dc0>), ('A', <pyspark.resultiterable.ResultIterable object at 0x7efec4d37020>)]


[('B', [4, 2]), ('A', [1, 5, 3])]

In [29]:
#aggregateByKey() allows you to return a different type than the input value type. It uses three functions:
data = [('A', 1), ('B', 2), ('A', 3), ('B', 4), ('A', 5)]
rdd = sc.parallelize(data)

# Calculate sum and count
zero_value = (0, 0)  # (sum, count)
seq_op = lambda acc, value: (acc[0] + value, acc[1] + 1)
comb_op = lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])

result = rdd.aggregateByKey(zero_value, seq_op, comb_op)

print(result)
result.collect()

PythonRDD[72] at RDD at PythonRDD.scala:58


[('B', (6, 2)), ('A', (9, 3))]

In [30]:
#combineByKey() is the most general of the per-key aggregation functions. It allows you to define how to create an initial accumulator, how to add new values to it, and how to merge accumulators.
data = [('A', 1), ('B', 2), ('A', 3), ('B', 4), ('A', 5)]
rdd = sc.parallelize(data)

def create_combiner(value):
    return (value, 1)  # (sum, count)

def merge_value(acc, value):
    return (acc[0] + value, acc[1] + 1)

def merge_combiners(acc1, acc2):
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

result = rdd.combineByKey(create_combiner, merge_value, merge_combiners)

print(result)
result.collect()

PythonRDD[78] at RDD at PythonRDD.scala:58


[('B', (6, 2)), ('A', (9, 3))]

## Actions

In [7]:
lines = sc.parallelize(["hello worl", "hi"])
words = lines.flatMap(lambda line: line.split(" "))
words

PythonRDD[8] at RDD at PythonRDD.scala:58

In [8]:
words.count()

3

In [12]:
words.collect()

['hello', 'worl', 'hi']

In [16]:
words.first()

'hello'

In [21]:
print(words.take(3))
print(words.take(2))

['hello', 'worl', 'hi']
['hello', 'worl']


In [32]:
distData.reduce(lambda a, b: a + b)  # 15

15

#### Persistence

In [ ]:
from pyspark import StorageLevel
frequently_used_rdd = text_file.filter(lambda line: "ERROR" in line)
frequently_used_rdd.persist(StorageLevel.MEMORY_AND_DISK)
# Available Storage levels in PySpark:
# StorageLevel.MEMORY_ONLY - Store in memory only
# StorageLevel.MEMORY_AND_DISK - Store in memory, spill to disk if needed
# StorageLevel.DISK_ONLY - Store on disk only
# StorageLevel.MEMORY_ONLY_SER - Store in memory in serialized format
# StorageLevel.MEMORY_AND_DISK_SER - Store in memory serialized, spill to disk
# StorageLevel.OFF_HEAP - Store in off-heap memory using Tachyon
# Alternative cache() method (uses MEMORY_ONLY by default)
frequently_used_rdd.cache()

#### Partitioning

In [4]:
from pyspark import SparkContext

key_value_rdd = sc.parallelize([("user1", "data1"), ("user2", "data2"), ("user1", "data3")])
# Hash partitioning with specific number of partitions
print(key_value_rdd)
key_value_rdd.glom().collect()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299


[[],
 [],
 [('user1', 'data1')],
 [],
 [],
 [('user2', 'data2')],
 [],
 [('user1', 'data3')]]

In [5]:
partitioned_rdd = key_value_rdd.partitionBy(4)  # Hash partitioning with 4 partitions
print(partitioned_rdd)
print(f"Number of partitions: {partitioned_rdd.getNumPartitions()}")
partitioned_rdd.glom().collect()

MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:170
Number of partitions: 4


[[('user2', 'data2')], [], [('user1', 'data1'), ('user1', 'data3')], []]

In [7]:
# Repartition to specific number of partitions
repartitioned_rdd = key_value_rdd.repartition(8)
print(repartitioned_rdd)
print(f"Number of partitions: {repartitioned_rdd.getNumPartitions()}")
repartitioned_rdd.glom().collect()

MapPartitionsRDD[17] at coalesce at <unknown>:0
Number of partitions: 8


[[('user1', 'data1'), ('user1', 'data3')],
 [],
 [('user2', 'data2')],
 [],
 [],
 [],
 [],
 []]

In [8]:
# Coalesce to reduce number of partitions (more efficient than repartition)
coalesced_rdd = key_value_rdd.coalesce(2)
print(coalesced_rdd)
print(f"Number of partitions: {coalesced_rdd.getNumPartitions()}")
coalesced_rdd.glom().collect()

CoalescedRDD[19] at coalesce at <unknown>:0
Number of partitions: 2


[[('user1', 'data1')], [('user2', 'data2'), ('user1', 'data3')]]

In [9]:
# Coalesce to reduce number of partitions (more efficient than repartition)
coalesced_rdd = key_value_rdd.coalesce(1)
print(coalesced_rdd)
print(f"Number of partitions: {coalesced_rdd.getNumPartitions()}")
coalesced_rdd.glom().collect()

CoalescedRDD[21] at coalesce at <unknown>:0
Number of partitions: 1


[[('user1', 'data1'), ('user2', 'data2'), ('user1', 'data3')]]

In [41]:
# Custom partitioning with sortByKey
sorted_rdd = key_value_rdd.sortByKey(numPartitions=4)
print(sorted_rdd)
print(f"Number of partitions: {sorted_rdd.getNumPartitions()}")
sorted_rdd.glom().collect()

PythonRDD[83] at RDD at PythonRDD.scala:58
Number of partitions: 4


[[('user1', 'data1'), ('user1', 'data3')], [], [('user2', 'data2')], []]